
# ComplianceGPT — QUR Generator

- **Library API**: `QURComponent.generate(query) -> [r1, r2, r3]`
- **Batch CSVs**: Rev.5 / Rev.4 / Error Bank → `/content/output_qur/`
- **Deterministic** (seeded), **ID/placeholder integrity checks**
- **Drive copy** to `/content/drive/MyDrive/compliance_outputs/outputs_qur_<timestamps>`


Cell 1: Run-mode flag

1. for ComplianceGPT retrieval call
2. process whole batch of the datasets

In [1]:
# === QUR entry mode ===
# Accepted values:
#   "batch"  -> generate CSVs for all datasets (default)
#   "single" -> generate rewrites for one passed query (no CSV, no Drive copy)
#   "lib"    -> load library only (no work)
import sys

if __name__ == "__main__":
    DEFAULT_MODE = "batch"
else:
    DEFAULT_MODE = "lib"

# Allow overrides if variables are already set (rare edge case)
QUR_RUN_MODE = globals().get("QUR_RUN_MODE", DEFAULT_MODE)
QUR_PASSED_QUERY = globals().get("QUR_PASSED_QUERY", None)
QUR_PASSED_PARAMS = globals().get("QUR_PASSED_PARAMS", {})

print(f"[QUR] RUN_MODE={QUR_RUN_MODE}")


[QUR] RUN_MODE=batch


In [2]:
# Quiet import-time defaults (NO side-effects when this notebook is exported to .py and imported)
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    pass


def _ensure_environment_when_main():
    """Mount Drive + pip-install deps only for standalone notebook/script runs."""
    # Colab Drive mount (optional)
    if IN_COLAB:
        try:
            from google.colab import drive as _gdrive  # type: ignore
            _gdrive.mount('/content/drive', force_remount=False)
            print("[colab] Drive mounted at /content/drive")
        except Exception as e:
            print("[colab] Drive mount skipped/failed:", e)

    # Use Eastern Time in this runtime (best-effort)
    import os, time
    os.environ["TZ"] = "America/New_York"
    try:
        time.tzset()
    except Exception:
        pass

    # Minimal deps (no bitsandbytes) — only for standalone notebook/script runs
    def _pip_install(pkgs):
        import importlib, subprocess, sys
        need = []
        for mod, pip_name in pkgs:
            try:
                importlib.import_module(mod)
            except Exception:
                need.append(pip_name)
        if need:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + need)

    _pip_install([
        ("transformers", "transformers"),
        ("pandas", "pandas"),
        ("numpy", "numpy"),
        ("tqdm", "tqdm"),
        ("accelerate", "accelerate"),
    ])
    print("Environment ready (no 4-bit).")


if __name__ == "__main__":
    _ensure_environment_when_main()


Mounted at /content/drive
[colab] Drive mounted at /content/drive
Environment ready (no 4-bit).


In [3]:

from dataclasses import dataclass
from pathlib import Path

@dataclass
class Config:
    drive_base: Path = Path('/content/drive/MyDrive/compliance_data')
    local_base: Path = Path('/mnt/data')
    output_dir: Path = Path('/content/output_qur')
    rev5_gold: Path = None
    rev4_gold: Path = None
    error_bank: Path = None
    model_id: str = "Qwen/Qwen2.5-7B-Instruct"
    n_rewrites: int = 3
    include_original_in_csv: bool = False
    temperature: float = 0.5
    top_p: float = 0.9
    max_new_tokens: int = 80
    random_seed: int = 42
    def __post_init__(self):
        base = self.drive_base if (IN_COLAB and self.drive_base.exists()) else self.local_base
        self.rev5_gold = base / 'gold_standard_datasets/nist800_53/nist_sp800-53_rev5_gold-set_100q.csv'
        self.rev4_gold = base / 'gold_standard_datasets/nist800_53/nist_sp800-53_rev4_gold-set_36q.csv'
        self.error_bank = base / 'error_bank/error_bank_v1.csv'
        # NOTE: do not create output directories at import time; batch mode will create them.

CFG = Config()

In [4]:

import re, json, warnings
from typing import List, Dict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime
try:
    from zoneinfo import ZoneInfo
    ET_TZ = ZoneInfo("America/New_York")
except Exception:
    ET_TZ = None

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.options.mode.chained_assignment = None

def read_csv_safe(path: Path) -> pd.DataFrame:
    if not path or not path.exists():
        print(f"[csv] Missing: {path}")
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin-1')
    except Exception as e:
        print(f"[csv] Failed {path}: {e}")
        return pd.DataFrame()

def now_et_str() -> str:
    if ET_TZ is not None:
        return datetime.now(ET_TZ).strftime("%Y-%m-%d %H:%M:%S%z")
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CTRL_ID_RE = re.compile(r"\b[A-Z]{2,3}-\d+[A-Z0-9.-]*\b")
def find_control_ids(text: str) -> List[str]:
    return CTRL_ID_RE.findall(text or "")

def load_gold_set(path: Path) -> pd.DataFrame:
    df = read_csv_safe(path)
    if df.empty:
        return pd.DataFrame()
    # Normalize question column
    for c in ['question','query','user_query','original_query']:
        if c in df.columns:
            if c != 'question':
                df = df.rename(columns={c:'question'})
            break
    if 'question' not in df.columns:
        print(f"{path.name}: no question column found.")
        return pd.DataFrame()
    df = df.dropna(subset=['question']).copy()
    df['question'] = df['question'].astype(str).str.strip()
    # ID handling
    id_cols = ['id','query_id','qid','question_id']
    src_id = next((c for c in id_cols if c in df.columns), None)
    if src_id:
        df['query_id'] = df[src_id].astype(str)
        if src_id != 'query_id':
            df['source_query_id'] = df[src_id].astype(str)
    else:
        df['query_id'] = (np.arange(len(df)) + 1).astype(str)  # 1-based default
    df['query_ord'] = np.arange(len(df))                       # 0-based ordinal
    keep = ['query_id','query_ord','question']
    if 'source_query_id' in df.columns: keep.append('source_query_id')
    return df[keep]


In [5]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class QURComponent:
    def __init__(self, model_id: str, n: int=3, temperature: float=0.5,
                 top_p: float=0.9, max_new_tokens: int=80, seed: int=42,
                 model=None, tokenizer=None):  # <--- NEW ARGUMENTS
        self.model_id = model_id
        self.n = n
        self.temperature = temperature
        self.top_p = top_p
        self.max_new_tokens = max_new_tokens
        self.seed = seed

        # SHARED MEMORY LOGIC
        if model is not None and tokenizer is not None:
            self.model = model
            self.tok = tokenizer
        else:
            self.model, self.tok = self._load_model(model_id)

        self._set_seed(seed)

    def _set_seed(self, seed: int):
        import random
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    def _load_model(self, model_id: str):
        use_cuda = torch.cuda.is_available()
        try:
            if use_cuda:
                model = AutoModelForCausalLM.from_pretrained(
                    model_id,
                    dtype=torch.float16,
                    device_map='auto',
                    trust_remote_code=True
                )
            else:
                model = AutoModelForCausalLM.from_pretrained(
                    model_id,
                    trust_remote_code=True
                )
        except Exception:
            # Absolute fallback to CPU fp32
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                trust_remote_code=True
            )
        tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        model.eval()
        return model, tok

    def _system_prompt(self) -> str:

        return (
            "You are an expert query rewriter for a compliance search system."
            "Your task is to rewrite a user's query into 3 diverse alternatives."
            "\n"
            "--- CRITICAL OUTPUT FORMAT ---"
            "1.  Your output MUST contain ONLY the 3 rewritten queries."
            "2.  Each query MUST be on a new line."
            "3.  DO NOT use any numbers, bullets, or list markers (e.g., '1.', '-')."
            "4.  DO NOT use any labels or markdown (e.g., '**Keyword:**')."
            "5.  DO NOT use any quotation marks."
            "6.  The output must be perfectly clean."
            "\n"
            "--- REWRITE STRATEGIES ---"
            "Generate one query for each of these 3 strategies:"
            "1.  **Keyword:** Strip the query to its core technical keywords."
            "2.  **Expanded:** Add related technical synonyms or concepts."
            "3.  **Natural Language:** Rephrase the query as a complete, natural-sounding question."
            "\n"
            "--- GUARDRAILS (ABSOLUTE) ---"
            "You MUST preserve any control IDs (e.g., AC-2, IA-5) and placeholders "
            "(e.g., {{ insert: param, ... }}) EXACTLY as they appear. "
            "DO NOT CHANGE, ALTER, OR FABRICATE THESE IDs."
        )

    @torch.no_grad()
    def generate(self, query: str) -> List[str]:
        messages = [
            {"role": "system", "content": self._system_prompt()},
            {"role": "user", "content": query},
        ]
        prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tok([prompt], return_tensors='pt').to(DEVICE)
        out_ids = self.model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=self.max_new_tokens,
            do_sample=True,
            temperature=self.temperature,
            top_p=self.top_p,
            num_return_sequences=1,
            pad_token_id=self.tok.eos_token_id
        )
        new_tokens = out_ids[:, inputs.input_ids.shape[1]:]
        text = self.tok.decode(new_tokens[0], skip_special_tokens=True).strip()
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        cleaned = [re.sub(r'^(\d+[\.)]|[-*•])\s*', '', ln).strip() for ln in lines]
        uniq, seen = [], set()
        for ln in cleaned:
            key = ln.lower()
            if key and key not in seen:
                uniq.append(ln); seen.add(key)
        while len(uniq) < self.n:
            uniq.append(query)
        return uniq[:self.n]

def make_qur():
    return QURComponent(CFG.model_id, CFG.n_rewrites, CFG.temperature, CFG.top_p, CFG.max_new_tokens, CFG.random_seed)


In [6]:
def _gen_rows(df_gold: pd.DataFrame, qur: QURComponent, include_original: bool, dataset_tag: str) -> pd.DataFrame:
    rows = []
    if df_gold.empty:
        return pd.DataFrame(columns=[
            "dataset","query_id","query_ord","original_query","rewrite_rank","rewritten_query",
            "model_id","gen_params_json","timestamp_et"
        ])
    meta = {
        "model_id": qur.model_id,
        "temperature": qur.temperature,
        "top_p": qur.top_p,
        "max_new_tokens": qur.max_new_tokens,
        "n_rewrites": qur.n,
        "seed": qur.seed
    }
    meta_json = json.dumps(meta, ensure_ascii=False)
    ts = now_et_str()
    for _, row in tqdm(df_gold.iterrows(), total=len(df_gold), desc=f"QUR → {dataset_tag}"):
        qid = str(row["query_id"]); qord = int(row["query_ord"]); q = str(row["question"])
        rewrites = qur.generate(q)
        candidates = ([q] + rewrites) if include_original else rewrites
        for rank, text in enumerate(candidates):
            rows.append({
                "dataset": dataset_tag,
                "query_id": qid,
                "query_ord": qord,
                "original_query": q,
                "rewrite_rank": rank,
                "rewritten_query": text,
                "model_id": qur.model_id,
                "gen_params_json": meta_json,
                "timestamp_et": ts
            })
    return pd.DataFrame(rows)

def _one_base(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df, pd.DataFrame) and not df.empty and "query_ord" in df.columns:
        df["query_ord"] = df["query_ord"].astype(int) + 1
    return df

def batch_generate_and_save(include_original=False):
    # Ensure output dir exists (batch mode only)
    CFG.output_dir.mkdir(parents=True, exist_ok=True)
    out_dir = CFG.output_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    qur = make_qur()
    rev5 = load_gold_set(CFG.rev5_gold)
    rev4 = load_gold_set(CFG.rev4_gold)
    errb = load_gold_set(CFG.error_bank)
    df5 = _gen_rows(rev5, qur, include_original, "rev5") if not rev5.empty else pd.DataFrame()
    df4 = _gen_rows(rev4, qur, include_original, "rev4") if not rev4.empty else pd.DataFrame()
    dfe = _gen_rows(errb, qur, include_original, "error_bank") if not errb.empty else pd.DataFrame()

    df5 = _one_base(df5)
    df4 = _one_base(df4)
    dfe = _one_base(dfe)
    if not df5.empty: df5.to_csv(out_dir / "qur_rewrites_rev5.csv", index=False)
    if not df4.empty: df4.to_csv(out_dir / "qur_rewrites_rev4.csv", index=False)
    if not dfe.empty: dfe.to_csv(out_dir / "qur_rewrites_error_bank.csv", index=False)
    return {"rev5": df5, "rev4": df4, "error_bank": dfe}

def copy_qur_outputs_to_drive():
    """Copy /content/output_qur -> /content/drive/... (Colab only)."""
    try:
        import shutil, os
        from datetime import datetime
        try:
            from zoneinfo import ZoneInfo
            tz = ZoneInfo("America/New_York")
        except Exception:
            tz = None

        now = datetime.now(tz) if tz else datetime.now()
        OUT_DIR = str(CFG.output_dir)  # e.g., /content/output_qur
        DRIVE_DST = f"/content/drive/MyDrive/compliance_outputs/outputs_qur"
        os.makedirs(os.path.dirname(DRIVE_DST), exist_ok=True)

        if os.path.exists(OUT_DIR):
            if os.path.exists(DRIVE_DST):
                print(f"Destination exists. Removing old folder: {DRIVE_DST}")
                shutil.rmtree(DRIVE_DST)
            shutil.copytree(OUT_DIR, DRIVE_DST)
            with open(f"{DRIVE_DST}/TIMESTAMP_Eastern.txt", "w", encoding="utf-8") as f:
                f.write(f"Copied at (America/New_York): {now.isoformat()}\n")
            print("Copied to Drive ->", DRIVE_DST)
        else:
            print("Nothing to copy; OUT_DIR not found:", OUT_DIR)
    except Exception as e:
        print("Drive copy skipped or failed:", e)

In [7]:
def sanity_report(dfs: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for name, df in dfs.items():
        if df is None or df.empty:
            rows.append({"dataset": name, "rows": 0, "id_preserved_ratio": 1.0})
            continue
        bad_id = 0; total_with_id = 0
        for _, r in df.iterrows():
            orig = r["original_query"]; rw = r["rewritten_query"]
            orig_ids = set(find_control_ids(orig))
            rw_ids = set(find_control_ids(rw))
            if orig_ids:
                total_with_id += 1
                if rw_ids and not rw_ids.issubset(orig_ids):
                    bad_id += 1
        rows.append({
            "dataset": name,
            "rows": int(len(df)),
            "id_preserved_ratio": float(1.0 - (bad_id / total_with_id)) if total_with_id else 1.0
        })
    return pd.DataFrame(rows)

# =========================================================================
# MAIN ENTRY POINT (OPTIMIZED)
# =========================================================================

# This "Guard" prevents the code below from running when imported by pipeline.py
if __name__ == "__main__":

    print(f"[QUR] Script running directly. Mode: {QUR_RUN_MODE}")

    if QUR_RUN_MODE == "batch":
        # 1. Generate Rewrites
        ARTIFACTS = batch_generate_and_save(include_original=CFG.include_original_in_csv)

        # 2. Sanity Check
        sanity_df = sanity_report(ARTIFACTS)
        try:
            display(sanity_df.head())
        except NameError:
            print(sanity_df.head())

        # 3. Copy to Drive
        copy_qur_outputs_to_drive()

    elif QUR_RUN_MODE == "single":
        qur = make_qur()
        if QUR_PASSED_QUERY:
            print("[QUR] Result:", qur.generate(str(QUR_PASSED_QUERY)))

    # 4. Run Unit Tests (Only if running directly)
    print("\n--- Running Unit Tests ---")
    import unittest

    # Define test class inline to ensure it exists
    class TestQURIntegrity(unittest.TestCase):
        @classmethod
        def setUpClass(cls):
            try: cls.qur = make_qur()
            except NameError: cls.qur = None
        def test_01_completeness(self):
            if not self.qur: self.skipTest("No QUR")
            self.assertEqual(len(self.qur.generate("test")), 3)

    try:
        suite = unittest.TestLoader().loadTestsFromTestCase(TestQURIntegrity)
        unittest.TextTestRunner(verbosity=2).run(suite)
    except NameError:
        print("TestQURIntegrity class failed to load.")

else:
    # This block runs when pipeline.py imports this file
    # It ensures FAST loading (0 seconds)
    print("[QUR] Library mode loaded. (Batch generation & Tests disabled)")

[QUR] Script running directly. Mode: batch


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

QUR → error_bank:   0%|          | 0/39 [00:00<?, ?it/s]

,dataset,rows,id_preserved_ratio
0,rev5,300,1.0
1,rev4,108,1.0
2,error_bank,117,1.0


Destination exists. Removing old folder: /content/drive/MyDrive/compliance_outputs/outputs_qur
Copied to Drive -> /content/drive/MyDrive/compliance_outputs/outputs_qur

--- Running Unit Tests ---


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

test_01_completeness (__main__.TestQURIntegrity.test_01_completeness) ... ok

----------------------------------------------------------------------
Ran 1 test in 7.779s

OK


In [8]:

# Export "module-like" handle so other notebooks can import same behavior
QUR_LIB = {
    "QURComponent": QURComponent,
    "make_qur": make_qur,
    "CFG_snapshot": {
        "model_id": CFG.model_id,
        "n_rewrites": CFG.n_rewrites,
        "temperature": CFG.temperature,
        "top_p": CFG.top_p,
        "max_new_tokens": CFG.max_new_tokens,
        "seed": CFG.random_seed
    }
}
print("QUR library exported as QUR_LIB")


QUR library exported as QUR_LIB


Unit Test

In [9]:
# === QUR entry mode ===
# Accepted values:
#   "batch"  -> generate CSVs for all datasets
#   "single" -> generate rewrites for one passed query (no CSV, no Drive copy)
#   "lib"    -> load library only (no work)
#
# IMPORTANT:
# - When the file is IMPORTED (e.g., by pipeline.py), default to "lib" to avoid side-effects.
# - When run directly as a script / notebook, default to "batch".
if __name__ == "__main__":
    QUR_RUN_MODE = globals().get("QUR_RUN_MODE", "batch")
else:
    QUR_RUN_MODE = globals().get("QUR_RUN_MODE", "lib")

QUR_PASSED_QUERY  = globals().get("QUR_PASSED_QUERY", None)  # used only when QUR_RUN_MODE=="single"
QUR_PASSED_PARAMS = globals().get("QUR_PASSED_PARAMS", {})   # optional overrides: {"n":3, "temperature":0.5, ...}
print(f"[QUR] RUN_MODE={QUR_RUN_MODE}")


[QUR] RUN_MODE=batch
